In [ ]:
from pathlib import Path
import sys

# --- Resolve repo root (works whether you run from repo root or within notebooks/) ---
REPO_ROOT = Path.cwd().resolve()
if not (REPO_ROOT / "src").exists():
    if (REPO_ROOT.parent / "src").exists():
        REPO_ROOT = REPO_ROOT.parent
    elif (REPO_ROOT.parent.parent / "src").exists():
        REPO_ROOT = REPO_ROOT.parent.parent

sys.path.insert(0, str(REPO_ROOT))
sys.path.insert(0, str(REPO_ROOT / "src"))

from IPython.display import display
from notebooks.set_up import (
    ensure_dirs,
    SYNTH_DIR,
    TABLE_DIR, FIG_DIR,
    STUDY_DESIGN_DIR, 
)

ensure_dirs()

# Inputs
base_path = str(SYNTH_DIR) + "/"
# Outputs
output_result = str(STUDY_DESIGN_DIR) + "/"  
#output_figure = str(FIG_DIR) + "/"

Estimators and evaluation utilities live in `src/causalmix/cate/`.

In [ ]:
# CATE estimators + evaluation
from causalmix.cate import *


# Application 3. Power analysis for prospective study design

In [ ]:
# functions used for power analysis
import numpy as np
import pandas as pd
import time

# outcome = "hosp_ed_any"
# treatment = "exp"
# effect_modifier = "cvd_pre"
# covariates = None
# data = df
# random_state = 36
# top_k = 10
# alpha = 0.05
# num_trees = 2000
# min_node_size = 80
# -----------------------
# Helper: resolve CVD column after preprocessing
# -----------------------
def resolve_feature_name(cols, target_name: str) -> str:
    """
    Try to find the intended feature column name in a robust way.
    - exact match
    - case-insensitive match
    - prefix match (e.g., one-hot variants like 'CVD_1')
    """
    if target_name in cols:
        return target_name
    lower_map = {c.lower(): c for c in cols}
    if target_name.lower() in lower_map:
        return lower_map[target_name.lower()]
    # prefix match
    # pref = target_name.lower()
    # candidates = [c for c in cols if c.lower().startswith(pref)]
    # if len(candidates) > 0:
    #     # deterministic choice: shortest name first
    #     candidates = sorted(candidates, key=lambda s: (len(s), s))
    #     return candidates[0]
    raise ValueError(f"Could not find effect modifier column '{target_name}' in columns.")

# -----------------------
# Core function: fit CF, screen via variable_importance, test CVD via BLP p-value
# -----------------------
def cf_detect_effect_modifier_via_blp(
    data: pd.DataFrame,
    outcome: str,
    treatment: str,
    effect_modifier: str,
    covariates=None,
    top_k: int = 5,
    alpha: float = 0.05,
    num_trees: int = 2000,
    min_node_size: int = 5,
    random_state: int = 36,
):
    """
    Returns dict with:
      - cvd_selected (0/1)
      - cvd_rank (1=most important)
      - cvd_importance
      - pval_cvd (NaN if not selected)
      - reject (0/1): reject H0: beta_CVD = 0 only if (CVD selected) & (p < alpha)
      - topk_vars (string)
    """
    # Import rpy2 inside (Spark-safe)
    import rpy2.robjects as ro
    from rpy2.robjects.packages import importr
    from rpy2.robjects.conversion import localconverter
    from rpy2.robjects import default_converter
    import rpy2.robjects.numpy2ri as numpy2ri

    grf = importr("grf")
    base = importr("base")

    if covariates is None:
        covariates = [c for c in data.columns if c not in [outcome, treatment]]

    # Resolve actual CVD column name post-preprocessing
    cvd_col = resolve_feature_name(covariates, effect_modifier)

    # Prepare arrays
    X_np = data[covariates].to_numpy(dtype=float)
    Y_np = data[outcome].to_numpy(dtype=float).ravel()
    W_np = data[treatment].to_numpy(dtype=float).ravel()

    # sanity checks
    if not set(np.unique(W_np)).issubset({0.0, 1.0}):
        raise ValueError("Treatment must be coded as 0/1 for grf::causal_forest.")
    if np.isnan(X_np).any() or np.isnan(Y_np).any() or np.isnan(W_np).any():
        raise ValueError("Missing values detected; grf does not like NAs without handling.")

    # Convert to R
    with localconverter(default_converter + numpy2ri.converter):
        Xr = ro.conversion.py2rpy(X_np)
    Xr = base.matrix(Xr, nrow=X_np.shape[0], ncol=X_np.shape[1])
    Yr = ro.FloatVector(Y_np)
    Wr = ro.FloatVector(W_np)

    # Fit causal forest
    ro.r["set.seed"](random_state)
    cf = grf.causal_forest(
        Xr, Yr, Wr,
        num_trees=num_trees,
        min_node_size=min_node_size,
        **{"num.threads": 1},   # avoid oversubscription on Spark workers
    )

    # Variable importance screening
    imp_r = grf.variable_importance(cf)  # length = p
    with localconverter(default_converter + numpy2ri.converter):
        imp = np.asarray(ro.conversion.rpy2py(imp_r), dtype=float)

    # ranks
    imp = np.asarray(imp).reshape(-1)

    order = np.argsort(-imp)           # shape (p,)
    ranks = np.empty(len(order), dtype=int)
    ranks[order] = np.arange(1, len(order) + 1)

    cvd_idx = covariates.index(cvd_col)
    cvd_rank = int(ranks[cvd_idx])
    cvd_importance = float(imp[cvd_idx])

    top_idx = order[:min(top_k, len(order))]
    top_vars = [covariates[i] for i in top_idx]
    cvd_selected = int(cvd_col in top_vars)

    pval_cvd = np.nan
    reject = 0

    # If selected, run best linear projection on top-K covariates and pull CVD p-value
    if cvd_selected:
        A_np = data[top_vars].to_numpy(dtype=float)

        with localconverter(default_converter + numpy2ri.converter):
            Ar = ro.conversion.py2rpy(A_np)
        Ar = base.matrix(Ar, nrow=A_np.shape[0], ncol=A_np.shape[1])

        # set column names so BLP prints/returns rows for actual variable names
        Ar = ro.r["colnames<-"](Ar, ro.StrVector(top_vars))

        blp = grf.best_linear_projection(
            cf,
            A=Ar,
            **{"vcov.type": "HC3"}  # default anyway; explicit for clarity
        )

        # best_linear_projection returns a coefficient table (matrix-like)
        rownames = list(ro.r["rownames"](blp))
        colnames = list(ro.r["colnames"](blp))

        with localconverter(default_converter + numpy2ri.converter):
            blp_mat = np.asarray(ro.conversion.rpy2py(blp), dtype=float)

        # locate p-value column (usually "Pr(>|t|)")
        pcol_candidates = [j for j, cn in enumerate(colnames) if ("Pr" in cn) or ("p" in cn.lower())]
        pcol = pcol_candidates[-1] if pcol_candidates else (blp_mat.shape[1] - 1)

        # locate CVD row
        if cvd_col not in rownames:
            # fallback: if rownames came through as A1, A2, ... use position
            # (CVD is in top_vars, so its position is known)
            pos = top_vars.index(cvd_col)  # 0-based among A columns
            # rows include intercept first, then A1..Ak
            ridx = 1 + pos
        else:
            ridx = rownames.index(cvd_col)

        pval_cvd = float(blp_mat[ridx, pcol])
        reject = int(pval_cvd < alpha)

    return {
    "cvd_col": cvd_col,
    "cvd_selected": cvd_selected,
    "cvd_rank": cvd_rank,
    "cvd_importance": cvd_importance,
    "pval_cvd": pval_cvd,
    "reject": reject,
    "topk_vars": "|".join(top_vars),
    }

# -----------------------
# An udpated Core function: fit CF, test CVD via BLP p-value among all covariates (not using variable importance)
# -----------------------
def cf_blp(
    data: pd.DataFrame,
    outcome: str,
    treatment: str,
    effect_modifier: str,
    covariates=None,
    alpha: float = 0.05,
    num_trees: int = 2000,
    min_node_size: int = 5,
    random_state: int = 36,
):
    """
    Best linear projection (BLP) on ALL covariates + variable-importance diagnostics.

    Returns dict with:
      - cvd_col
      - cvd_rank (1=most important by variable_importance)
      - cvd_importance
      - pval_cvd
      - reject (0/1): pval_cvd < alpha
    """
    import numpy as np
    import rpy2.robjects as ro
    from rpy2.robjects.packages import importr
    from rpy2.robjects.conversion import localconverter
    from rpy2.robjects import default_converter
    import rpy2.robjects.numpy2ri as numpy2ri

    grf = importr("grf")
    base = importr("base")

    if covariates is None:
        covariates = [c for c in data.columns if c not in [outcome, treatment]]

    # Resolve actual effect-modifier column name post-preprocessing
    cvd_col = resolve_feature_name(covariates, effect_modifier)

    # Prepare arrays
    X_np = data[covariates].to_numpy(dtype=float)
    Y_np = data[outcome].to_numpy(dtype=float).ravel()
    W_np = data[treatment].to_numpy(dtype=float).ravel()

    # Sanity checks
    if not set(np.unique(W_np)).issubset({0.0, 1.0}):
        raise ValueError("Treatment must be coded as 0/1 for grf::causal_forest.")
    if np.isnan(X_np).any() or np.isnan(Y_np).any() or np.isnan(W_np).any():
        raise ValueError("Missing values detected; handle NAs before grf.")

    # Convert to R
    with localconverter(default_converter + numpy2ri.converter):
        Xr = ro.conversion.py2rpy(X_np)
    Xr = base.matrix(Xr, nrow=X_np.shape[0], ncol=X_np.shape[1])
    Yr = ro.FloatVector(Y_np)
    Wr = ro.FloatVector(W_np)

    # Fit causal forest
    ro.r["set.seed"](random_state)
    cf = grf.causal_forest(
        Xr, Yr, Wr,
        num_trees=num_trees,
        min_node_size=min_node_size,
        **{"num.threads": 1},
    )

    # ---------- Variable importance diagnostics ----------
    imp_r = grf.variable_importance(cf)
    with localconverter(default_converter + numpy2ri.converter):
        imp = np.asarray(ro.conversion.rpy2py(imp_r), dtype=float)
    imp = np.asarray(imp).reshape(-1)  # critical fix: force 1D

    order = np.argsort(-imp)           # descending indices, shape (p,)
    ranks = np.empty(len(order), dtype=int)
    ranks[order] = np.arange(1, len(order) + 1)

    cvd_idx = covariates.index(cvd_col)
    cvd_rank = int(ranks[cvd_idx])
    cvd_importance = float(imp[cvd_idx])

    # ---------- Best linear projection on ALL covariates ----------
    # A matrix is the covariate design for the linear projection
    A_np = X_np  # all covariates
    with localconverter(default_converter + numpy2ri.converter):
        Ar = ro.conversion.py2rpy(A_np)
    Ar = base.matrix(Ar, nrow=A_np.shape[0], ncol=A_np.shape[1])
    Ar = ro.r["colnames<-"](Ar, ro.StrVector(covariates))

    blp = grf.best_linear_projection(cf, A=Ar, **{"vcov.type": "HC3"})

    rownames = list(ro.r["rownames"](blp))
    colnames = list(ro.r["colnames"](blp))
    with localconverter(default_converter + numpy2ri.converter):
        blp_mat = np.asarray(ro.conversion.rpy2py(blp), dtype=float)

    # Locate p-value column (usually "Pr(>|t|)")
    pcol_candidates = [j for j, cn in enumerate(colnames) if ("Pr" in str(cn)) or ("p" in str(cn).lower())]
    pcol = pcol_candidates[-1] if pcol_candidates else (blp_mat.shape[1] - 1)

    # Locate CVD row
    if cvd_col in rownames:
        ridx = rownames.index(cvd_col)
    else:
        # fallback if rownames are like "A1", "A2", ...
        pos = covariates.index(cvd_col)
        ridx = 1 + pos  # intercept is row 0

    pval_cvd = float(blp_mat[ridx, pcol])
    reject = int(pval_cvd < alpha)

    return {
        "cvd_col": cvd_col,
        "cvd_rank": cvd_rank,
        "cvd_importance": cvd_importance,
        "pval_cvd": pval_cvd,
        "reject": reject,
    }


In [0]:
# First analysis: run the power analysis with screening of variables using variable importance [not run or used, as the curve won't be smooth if screening variables before the blp]
# Fixed version: Load data on driver, avoid spark inside UDF
import numpy as np
import pandas as pd
import os

# -----------------------
# User settings
# -----------------------
n_reps = 100
delta_name_pattern = "df_n{n}_gen_{rep}"

target_concurrency = 12 # there are 16 cores

OUTCOME = "hosp_ed_any"
TREATMENT = "exp"
EFFECT_MODIFIER = "cvd_pre"
TOP_K = 10
ALPHA = 0.05
TARGET_POWER = 0.80

N_GRID = [500, 1000, 2000, 5000]# [20000, 50000, 2000, 1000, 500]

BEST_NUM_TREES = 2000
BEST_MIN_NODE_SIZE = 80

# Create output directory
os.makedirs(output_result, exist_ok=True)

# -----------------------
# 1) Load ALL data on driver (avoid spark in UDF)
# -----------------------
data_parts = []

for rep in range(n_reps):
    for n in N_GRID:
        # Load dataset on driver
        pdf = spark.read.format("delta").load(
            base_path + delta_name_pattern.format(n=n, rep=rep)
        ).toPandas()
        pdf = pdf.reset_index(drop=True)
        pdf["idx"] = np.arange(len(pdf), dtype=int)
        pdf["rep"] = rep
        pdf["n"] = n
        data_parts.append(pdf)

data_all_pdf = pd.concat(data_parts, ignore_index=True)
data_sdf = spark.createDataFrame(data_all_pdf)

print(f"Loaded {len(data_parts)} datasets (n_reps={n_reps}, n_grid={N_GRID})")

# -----------------------
# 2) Pandas UDF per (rep, n): NO spark usage inside
# -----------------------

Loaded 5 datasets (n_reps=5, n_grid=[1000])

Power by n:


n,power,cvd_selected_rate,mean_cvd_rank,reps
1000,0.0,0.0,15.0,5



Minimum sample size n* achieving power >= 0.80: None


In [0]:
# Second analysis: run the power analysis using BLP on all covariates (without variable selection)
import os
import numpy as np
import pandas as pd

n_reps = 100
delta_name_pattern = "df_n{n}_gen_{rep}"

target_concurrency = 12 # there are 16 cores

OUTCOME = "hosp_ed_any"
TREATMENT = "exp"
EFFECT_MODIFIER = "cvd_pre"
ALPHA = 0.05
TARGET_POWER = 0.80

N_GRID = [200, 500, 1000, 2000, 5000]

BEST_NUM_TREES = 2000
BEST_MIN_NODE_SIZE = 80

# Create output directory
os.makedirs(output_result, exist_ok=True)

# -----------------------
# 1) Load ALL data on driver (avoid spark in UDF)
# -----------------------
data_parts = []

for rep in range(n_reps):
    for n in N_GRID:
        pdf = spark.read.format("delta").load(
            base_path + delta_name_pattern.format(n=n, rep=rep)
        ).toPandas()
        pdf = pdf.reset_index(drop=True)
        pdf["idx"] = np.arange(len(pdf), dtype=int)
        pdf["rep"] = rep
        pdf["n"] = n
        data_parts.append(pdf)

data_all_pdf = pd.concat(data_parts, ignore_index=True)
data_sdf = spark.createDataFrame(data_all_pdf)

print(f"Loaded {len(data_parts)} datasets (n_reps={n_reps}, n_grid={N_GRID})")

# -----------------------
# 2) Pandas UDF per (rep, n): NO spark usage inside
# -----------------------
def eval_one_job(pdf: pd.DataFrame) -> pd.DataFrame:
    rep = int(pdf["rep"].iloc[0])
    n = int(pdf["n"].iloc[0])

    try:
        df = pdf.sort_values("idx").drop(columns=["idx", "rep", "n"])

        # Preprocess
        df_onehot = preprocess_data(df, dummy_code=False)
        covariates = [c for c in df_onehot.columns if c not in [OUTCOME, TREATMENT]]

        out = cf_blp(
            df_onehot,
            outcome=OUTCOME,
            treatment=TREATMENT,
            effect_modifier=EFFECT_MODIFIER,
            covariates=covariates,
            alpha=ALPHA,
            num_trees=BEST_NUM_TREES,
            min_node_size=BEST_MIN_NODE_SIZE,
            random_state=36 + 10_000 * rep + n,
        )

        row = {
            "rep": rep,
            "n": n,
            "cvd_rank": float(out["cvd_rank"]),
            "cvd_importance": float(out["cvd_importance"]),
            "pval_cvd": float(out["pval_cvd"]) if np.isfinite(out["pval_cvd"]) else np.nan,
            "reject": int(out["reject"]),
            "error": "",
        }
        return pd.DataFrame([row])

    except Exception as e:
        row = {
            "rep": rep,
            "n": n,
            "cvd_rank": np.nan,
            "cvd_importance": np.nan,
            "pval_cvd": np.nan,
            "reject": 0,
            "error": str(e),
        }
        return pd.DataFrame([row])

schema = """
rep int,
n int,
cvd_rank double,
cvd_importance double,
pval_cvd double,
reject int,
error string
"""

# -----------------------
# 3) Run distributed evaluation
# -----------------------
per_rep_spark = (
    data_sdf
    .repartition(target_concurrency, "rep")
    .groupBy("rep", "n")
    .applyInPandas(eval_one_job, schema=schema)
)

per_rep_df = per_rep_spark.toPandas()

# -----------------------
# 4) Summarize power vs n
# -----------------------
valid = per_rep_df[per_rep_df["error"] == ""].copy()

power_df = (
    valid.groupby("n", as_index=False)
    .agg(
        power=("reject", "mean"),
        mean_cvd_rank=("cvd_rank", "mean"),
        mean_cvd_importance=("cvd_importance", "mean"),
        reps=("rep", "nunique"),
    )
    .sort_values("n")
)

eligible = power_df[power_df["power"] >= TARGET_POWER]
n_star = int(eligible["n"].min()) if len(eligible) else None

print("\nPower by n:")
print(power_df)  # replaced display()

print(f"\nMinimum sample size n* achieving power >= {TARGET_POWER:.2f}: {n_star}")

# Save (optional)
per_rep_df.to_csv(output_result + "per_rep_power_allcov.csv", index=False)
power_df.to_csv(output_result + "power_by_n_allcov.csv", index=False)
# it took 34 minutes to run 100 repetitions for all datasets

Loaded 500 datasets (n_reps=100, n_grid=[200, 500, 1000, 2000, 5000])

Power by n:


n,power,mean_cvd_rank,mean_cvd_importance,reps
200,0.13,15.0,0.0,100
500,0.54,15.0,0.0,100
1000,0.67,15.0,0.0,100
2000,0.94,8.28,0.012069197488658635,100
5000,1.0,1.0,0.5764642794208611,100



Minimum sample size n* achieving power >= 0.80: 2000


In [0]:
print(per_rep_df)  # replaced display()

rep,n,cvd_rank,cvd_importance,pval_cvd,reject,error
28,200,15.0,0.0,0.4786797417214851,0,
28,500,15.0,0.0,0.013020881202092003,1,
28,1000,15.0,0.0,0.007017864777076718,1,
28,2000,9.0,0.0017578554163920016,0.016226451864776968,1,
28,5000,1.0,0.5076564162832782,6.798810535555303E-7,1,
30,200,15.0,0.0,0.1806203303740973,0,
30,500,15.0,0.0,0.39990497454705876,0,
30,1000,15.0,0.0,0.0029672139539820373,1,
30,2000,10.0,0.0014055808392000877,0.002414991598877044,1,
30,5000,1.0,0.4174701601124432,2.30589966049657E-4,1,
